# 05 - Inventory Analysis

## Problem 4: Inventory Optimization

This notebook uses **Databricks SQL as the primary analysis layer**, following the capstone requirements.

**Input Silver tables**
- `inventory_silver`
- `sales_silver`
- `products_silver`
- `stores_silver`
- `suppliers_silver`
- `purchase_orders_silver`

The notebook answers Questions 22–27 from the capstone. Where the source data does not contain a field required for an exact business metric, the limitation is documented instead of inventing data.

## Q22. Which products are currently out of stock or low stock (< 50 units)?

Business rule from the capstone:
- **Out of stock:** `quantity_on_hand = 0`
- **Low stock:** `quantity_on_hand < 50`

We use `LEFT JOIN` so that an inventory record is still visible even if product/store descriptive information is missing.

In [0]:
%sql

SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    CASE
        WHEN i.quantity_on_hand = 0 THEN 'Out of Stock'
        WHEN i.quantity_on_hand < 50 THEN 'Low Stock'
        ELSE 'Normal Stock'
    END AS stock_status
FROM inventory_silver i
LEFT JOIN stores_silver s
    ON i.store_id = s.store_id
LEFT JOIN products_silver p
    ON i.product_id = p.product_id
WHERE i.quantity_on_hand < 50
ORDER BY
    i.quantity_on_hand ASC,
    i.store_id,
    i.product_id;


## Q23. Which products have the highest sales velocity?

**Sales velocity assumption:** average quantity sold per day during 2024.

Formula:

`Sales Velocity = Total Quantity Sold / 365`

The capstone does not prescribe a specific velocity formula, so this definition is made explicit for reproducibility.

In [0]:
%sql

SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(sa.quantity_sold) AS total_quantity_sold,
    ROUND(SUM(sa.quantity_sold) / 365.0, 2) AS avg_daily_sales
FROM sales_silver sa
INNER JOIN products_silver p
    ON sa.product_id = p.product_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY
    p.product_id,
    p.product_name,
    p.category
ORDER BY
    avg_daily_sales DESC,
    total_quantity_sold DESC
LIMIT 10;


## Q24. Which products are slow-moving and overstock candidates?

The capstone does not provide numerical thresholds for "slow-moving" or "overstock".

For a reproducible analysis, this query uses:
- **Slow-moving:** product's sales velocity is below the median product velocity.
- **Overstock candidate:** current store inventory is above the median inventory quantity for that product.

This is an analytical assumption, not a fixed business rule from the source document.

In [0]:
%sql

WITH product_velocity AS (
    SELECT
        product_id,
        SUM(quantity_sold) / 365.0 AS avg_daily_sales
    FROM sales_silver
    WHERE YEAR(sale_date) = 2024
    GROUP BY product_id
),

velocity_threshold AS (
    SELECT
        percentile_approx(avg_daily_sales, 0.5) AS median_daily_sales
    FROM product_velocity
),

store_inventory AS (
    SELECT
        i.store_id,
        i.product_id,
        i.quantity_on_hand
    FROM inventory_silver i
),

inventory_threshold AS (
    SELECT
        percentile_approx(quantity_on_hand, 0.5) AS median_inventory
    FROM store_inventory
)

SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    ROUND(v.avg_daily_sales, 2) AS avg_daily_sales,
    CASE
        WHEN v.avg_daily_sales < vt.median_daily_sales
             AND i.quantity_on_hand > it.median_inventory
        THEN 'Slow-Moving & Overstock Candidate'
        WHEN v.avg_daily_sales < vt.median_daily_sales
        THEN 'Slow-Moving'
        WHEN i.quantity_on_hand > it.median_inventory
        THEN 'Potential Overstock'
    END AS inventory_flag
FROM store_inventory i
INNER JOIN product_velocity v
    ON i.product_id = v.product_id
CROSS JOIN velocity_threshold vt
CROSS JOIN inventory_threshold it
INNER JOIN stores_silver s
    ON i.store_id = s.store_id
INNER JOIN products_silver p
    ON i.product_id = p.product_id
WHERE v.avg_daily_sales < vt.median_daily_sales
   OR i.quantity_on_hand > it.median_inventory
ORDER BY
    inventory_flag,
    i.quantity_on_hand DESC;


## Q25. What is the average days to deliver for each supplier?

Delivery time is calculated as:

`delivery_date - order_date`

Cancelled/uncompleted orders with a null `delivery_date` are excluded from the average because there is no completed delivery duration.

In [0]:
%sql

SELECT
    s.supplier_id,
    s.supplier_name,
    COUNT(po.po_id) AS completed_orders,
    ROUND(
        AVG(DATEDIFF(po.delivery_date, po.order_date)),
        2
    ) AS avg_delivery_days
FROM purchase_orders_silver po
INNER JOIN suppliers_silver s
    ON po.supplier_id = s.supplier_id
WHERE po.delivery_date IS NOT NULL
GROUP BY
    s.supplier_id,
    s.supplier_name
ORDER BY
    avg_delivery_days ASC;


## Q26. Which suppliers are most reliable (on-time delivery rate)?

### Important data limitation

The available purchase-order data contains `order_date` and `delivery_date`, but **does not contain a promised/expected delivery date**.

Therefore, a true on-time delivery rate cannot be calculated from the available data without inventing a due-date rule.

Instead, this query provides a transparent supplier performance scorecard using:
- completed delivery rate
- average delivery days
- completed order count

A true on-time rate can be added later if an expected delivery date is available.

In [0]:
%sql

SELECT
    s.supplier_id,
    s.supplier_name,
    COUNT(po.po_id) AS total_orders,
    SUM(
        CASE
            WHEN po.delivery_date IS NOT NULL THEN 1
            ELSE 0
        END
    ) AS completed_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN po.delivery_date IS NOT NULL THEN 1
                ELSE 0
            END
        ) / COUNT(po.po_id),
        2
    ) AS completion_rate_pct,
    ROUND(
        AVG(
            CASE
                WHEN po.delivery_date IS NOT NULL
                THEN DATEDIFF(po.delivery_date, po.order_date)
            END
        ),
        2
    ) AS avg_delivery_days
FROM purchase_orders_silver po
INNER JOIN suppliers_silver s
    ON po.supplier_id = s.supplier_id
GROUP BY
    s.supplier_id,
    s.supplier_name
ORDER BY
    completion_rate_pct DESC,
    avg_delivery_days ASC;


## Q27. What is the optimal reorder point for each product-store combination?

The capstone requires **Python logic** for the final reorder-point calculation. This SQL query prepares the required analytical inputs.

For a practical baseline, we use:

`Reorder Point = Average Daily Sales × Average Supplier Lead Time`

Assumptions:
- Average daily sales = 2024 quantity sold / 365.
- Supplier lead time = average completed delivery days for the product's supplier.
- `products_silver.supplier_id` identifies the supplier.
- This is a baseline reorder point and does **not** include safety stock because the capstone data does not provide demand variability or a service-level target.

The resulting table is ready for the later Python `inventory_optimizer.py` step.

In [0]:
%sql

WITH sales_velocity AS (
    SELECT
        store_id,
        product_id,
        SUM(quantity_sold) / 365.0 AS avg_daily_sales
    FROM sales_silver
    WHERE YEAR(sale_date) = 2024
    GROUP BY
        store_id,
        product_id
),

supplier_lead_time AS (
    SELECT
        supplier_id,
        AVG(DATEDIFF(delivery_date, order_date)) AS avg_lead_time_days
    FROM purchase_orders_silver
    WHERE delivery_date IS NOT NULL
    GROUP BY supplier_id
)

SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    p.supplier_id,
    ROUND(COALESCE(v.avg_daily_sales, 0), 2) AS avg_daily_sales,
    ROUND(COALESCE(sl.avg_lead_time_days, 0), 2) AS avg_supplier_lead_time_days,
    ROUND(
        COALESCE(v.avg_daily_sales, 0)
        * COALESCE(sl.avg_lead_time_days, 0),
        2
    ) AS baseline_reorder_point
FROM inventory_silver i
INNER JOIN stores_silver s
    ON i.store_id = s.store_id
INNER JOIN products_silver p
    ON i.product_id = p.product_id
LEFT JOIN sales_velocity v
    ON i.store_id = v.store_id
   AND i.product_id = v.product_id
LEFT JOIN supplier_lead_time sl
    ON p.supplier_id = sl.supplier_id
ORDER BY
    i.store_id,
    baseline_reorder_point DESC;


## Problem 4 SQL Concepts Used

- `JOIN` / `LEFT JOIN`
- `GROUP BY`
- `SUM()` and `AVG()`
- `COUNT()`
- `CASE`
- `DATEDIFF()`
- `COALESCE()`
- `percentile_approx()`
- CTEs using `WITH`
- Business-rule classification
- Analytical preparation for Python-based optimization

### Next deliverables

This notebook provides the SQL analysis foundation for:
- `inventory_analysis.sql`
- `inventory_optimizer.py`
- Reorder recommendations
- Supplier performance scorecard

The Python optimizer should use the Q27 result to apply the final reorder/recommendation logic.